# Neo4j Graph Import — Final Forensic Model

This notebook imports the validated endpoint-security data into Neo4j.

The graph model is:

(:Endpoint)-[:OBSERVED]->(:Artifact)

Endpoint nodes represent every endpoint present in the raw security telemetry.

Artifact nodes are identified by SHA-256.

OBSERVED relationships contain endpoint-specific observation properties,
including explicit malicious-security evidence.

The model distinguishes:

- complete endpoint population
- endpoint-artifact observations
- explicit malicious detections
- activity-based investigation metrics

In [1]:
# Import getpass so the Neo4j password is entered securely rather than
# stored directly inside the notebook source.
import getpass

# Import pathlib for portable project-relative file paths.
from pathlib import Path

# Import pandas for loading and validating the exported CSV files.
import pandas as pd

# Import the official Neo4j Python driver.
from neo4j import GraphDatabase


# Resolve paths relative to the notebook's location.
DATA_DIR = Path("../data")

ENDPOINT_SECURITY_PATH = DATA_DIR / "endpoint_security.csv"
ARTIFACT_TABLE_PATH = DATA_DIR / "artifact_level_table.csv"
EDGE_TABLE_PATH = DATA_DIR / "endpoint_artifact_edges.csv"


# Neo4j connection configuration used by the project.
URI = "neo4j://127.0.0.1:7687"
USERNAME = "neo4j"
DATABASE = "indianarmygraph"


print("Endpoint security:", ENDPOINT_SECURITY_PATH.resolve())
print("Artifact table:", ARTIFACT_TABLE_PATH.resolve())
print("Edge table:", EDGE_TABLE_PATH.resolve())
print("Neo4j database:", DATABASE)

Endpoint security: C:\Users\Asus\Desktop\web_activity\data\endpoint_security.csv
Artifact table: C:\Users\Asus\Desktop\web_activity\data\artifact_level_table.csv
Edge table: C:\Users\Asus\Desktop\web_activity\data\endpoint_artifact_edges.csv
Neo4j database: indianarmygraph


In [2]:
# Prompt securely for the Neo4j password.
PASSWORD = getpass.getpass("Enter the Neo4j password: ")

# Create the Neo4j driver used throughout the notebook.
driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD),
)

# Verify connectivity before modifying the database.
driver.verify_connectivity()

print("Neo4j connection successful.")

Neo4j connection successful.


## Load and validate source data

The endpoint list is taken from the complete raw security telemetry.

The endpoint-artifact relationship table was generated by the corrected
artifact-level notebook and contains both ordinary observations and
explicit malicious-security evidence.

In [3]:
# Load the complete raw security telemetry.
security_df = pd.read_csv(ENDPOINT_SECURITY_PATH)

# Load the validated endpoint-artifact relationship table.
edge_df = pd.read_csv(EDGE_TABLE_PATH)

# Load the artifact-level table containing artifact-wide properties.
artifact_df = pd.read_csv(ARTIFACT_TABLE_PATH)


# Extract every endpoint present in the raw telemetry.
all_endpoint_ids = sorted(
    security_df["endpoint_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)


print("=" * 70)
print("SOURCE DATA VALIDATION")
print("=" * 70)

print("Raw security rows:", len(security_df))
print("Raw endpoints:", len(all_endpoint_ids))
print("Endpoint-artifact relationships:", len(edge_df))
print("Artifact rows:", len(artifact_df))

SOURCE DATA VALIDATION
Raw security rows: 23540
Raw endpoints: 247
Endpoint-artifact relationships: 72
Artifact rows: 60


In [4]:
# Verify that the relationship table contains the columns required
# by the final Neo4j relationship model.
required_edge_columns = {
    "endpoint_id",
    "sha256",
    "file_names",
    "record_count",
    "first_event",
    "last_event",
    "file_name_count",
    "malicious_detected",
    "malicious_detection_count",
    "malicious_file_names",
    "first_malicious_detection",
    "last_malicious_detection",
    "first_quarantine_time",
    "last_quarantine_time",
}

missing_edge_columns = (
    required_edge_columns - set(edge_df.columns)
)

if missing_edge_columns:
    raise ValueError(
        "The relationship CSV is missing required columns: "
        f"{sorted(missing_edge_columns)}"
    )


# Verify that every raw endpoint is represented in the endpoint population.
assert len(all_endpoint_ids) == 247, (
    f"Expected 247 raw endpoints, found {len(all_endpoint_ids)}."
)

# Verify the relationship count established by the previous notebook.
assert len(edge_df) == 72, (
    f"Expected 72 endpoint-artifact relationships, found {len(edge_df)}."
)

print("Source schema validation passed.")

Source schema validation passed.


## Reset existing graph

The existing graph was built from the previous malicious-only scope.

This cell clears the dedicated `indianarmygraph` database so the corrected
complete endpoint model can be imported without leaving stale nodes or
relationships behind.

This operation is destructive and should only be run against the project's
dedicated investigation database.

In [5]:
# Remove the previous graph so stale malicious-only nodes and relationships
# cannot remain after the corrected import.
reset_query = """
MATCH (n)
DETACH DELETE n
"""

driver.execute_query(
    reset_query,
    database_=DATABASE,
)

print("Existing graph cleared.")

Existing graph cleared.


In [6]:
# Enforce uniqueness for endpoint identifiers.
# The constraint also provides indexed lookup support for Endpoint nodes.
driver.execute_query(
    """
    CREATE CONSTRAINT endpoint_id_unique IF NOT EXISTS
    FOR (e:Endpoint)
    REQUIRE e.endpoint_id IS UNIQUE
    """,
    database_=DATABASE,
)

# Enforce uniqueness for SHA-256 artifact identifiers.
# This guarantees one Artifact node per unique SHA-256.
driver.execute_query(
    """
    CREATE CONSTRAINT artifact_sha256_unique IF NOT EXISTS
    FOR (a:Artifact)
    REQUIRE a.sha256 IS UNIQUE
    """,
    database_=DATABASE,
)

print("Neo4j constraints verified.")

Neo4j constraints verified.


In [7]:
# Import every endpoint found in the raw telemetry.
# This deliberately includes endpoints that have no parseable artifact.
endpoint_records = [
    {"endpoint_id": endpoint_id}
    for endpoint_id in all_endpoint_ids
]


endpoint_query = """
UNWIND $endpoints AS row

MERGE (e:Endpoint {
    endpoint_id: row.endpoint_id
})
"""


driver.execute_query(
    endpoint_query,
    endpoints=endpoint_records,
    database_=DATABASE,
)

print(
    f"Imported {len(endpoint_records)} Endpoint nodes."
)

Imported 247 Endpoint nodes.


In [8]:
# Convert pipe-delimited CSV list fields back into Python lists
# before sending them to Neo4j.
def split_pipe_list(value):
    """Convert a pipe-delimited CSV field into a clean Python list."""

    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    return [
        item.strip()
        for item in value.split("|")
        if item.strip()
    ]


# Normalize the artifact table's list-valued columns.
artifact_df["file_names"] = (
    artifact_df["file_names"]
    .apply(split_pipe_list)
)

artifact_df["endpoint_ids"] = (
    artifact_df["endpoint_ids"]
    .apply(split_pipe_list)
)


print(
    "Artifacts prepared:",
    len(artifact_df),
)

Artifacts prepared: 60


In [9]:
# Convert relationship list-valued columns back into Python lists.
edge_df["file_names"] = (
    edge_df["file_names"]
    .apply(split_pipe_list)
)

edge_df["malicious_file_names"] = (
    edge_df["malicious_file_names"]
    .apply(split_pipe_list)
)


# Normalize the malicious flag because CSV loading may represent booleans
# as strings depending on how the file was written.
edge_df["malicious_detected"] = (
    edge_df["malicious_detected"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)


# Normalize integer relationship properties.
edge_df["record_count"] = (
    edge_df["record_count"]
    .fillna(0)
    .astype(int)
)

edge_df["file_name_count"] = (
    edge_df["file_name_count"]
    .fillna(0)
    .astype(int)
)

edge_df["malicious_detection_count"] = (
    edge_df["malicious_detection_count"]
    .fillna(0)
    .astype(int)
)


# Convert timestamp columns into Python datetime objects.
# Neo4j accepts Python datetime values directly.
datetime_columns = [
    "first_event",
    "last_event",
    "first_malicious_detection",
    "last_malicious_detection",
    "first_quarantine_time",
    "last_quarantine_time",
]

for column in datetime_columns:
    edge_df[column] = pd.to_datetime(
        edge_df[column],
        errors="coerce",
    )


print("Relationship data prepared.")

Relationship data prepared.


In [10]:
# Convert pandas artifact records into ordinary Python dictionaries.
artifact_records = artifact_df.to_dict("records")


artifact_query = """
UNWIND $artifacts AS row

MERGE (a:Artifact {
    sha256: row.sha256
})

SET
    a.file_names = row.file_names,
    a.file_name_count = row.file_name_count,
    a.record_count = row.record_count,
    a.first_event = row.first_event,
    a.last_event = row.last_event,
    a.endpoint_count = row.endpoint_count,
    a.endpoint_ids = row.endpoint_ids
"""


driver.execute_query(
    artifact_query,
    artifacts=artifact_records,
    database_=DATABASE,
)

print(
    f"Imported {len(artifact_records)} Artifact nodes."
)

Imported 60 Artifact nodes.


In [11]:
# Convert pandas datetime values into Python datetime values or None.
# This avoids passing pandas NaT values into the Neo4j driver.
def neo4j_datetime(value):
    """Convert a pandas timestamp into a Neo4j-compatible Python datetime."""

    if pd.isna(value):
        return None

    return value.to_pydatetime()


for column in datetime_columns:
    edge_df[column] = (
        edge_df[column]
        .apply(neo4j_datetime)
    )


# Replace pandas/numpy scalar values with native Python types where needed.
edge_df["endpoint_id"] = (
    edge_df["endpoint_id"]
    .astype(str)
)

edge_df["sha256"] = (
    edge_df["sha256"]
    .astype(str)
    .str.lower()
)

edge_df["record_count"] = (
    edge_df["record_count"]
    .astype(int)
)

edge_df["file_name_count"] = (
    edge_df["file_name_count"]
    .astype(int)
)

edge_df["malicious_detection_count"] = (
    edge_df["malicious_detection_count"]
    .astype(int)
)

edge_df["malicious_detected"] = (
    edge_df["malicious_detected"]
    .astype(bool)
)


edge_records = edge_df.to_dict("records")

print(
    "Prepared relationship records:",
    len(edge_records),
)

Prepared relationship records: 72


In [12]:
# Create one OBSERVED relationship for each unique endpoint-artifact pair.
#
# Malicious evidence is stored on the relationship because the security
# classification belongs to the observation context of that endpoint-artifact
# pair rather than globally to the Artifact node.
relationship_query = """
UNWIND $edges AS row

MATCH (e:Endpoint {
    endpoint_id: row.endpoint_id
})

MATCH (a:Artifact {
    sha256: row.sha256
})

MERGE (e)-[r:OBSERVED]->(a)

SET
    r.record_count = row.record_count,
    r.first_event = row.first_event,
    r.last_event = row.last_event,

    r.malicious_detected = row.malicious_detected,
    r.malicious_detection_count = row.malicious_detection_count,
    r.malicious_file_names = row.malicious_file_names,

    r.first_malicious_detection = row.first_malicious_detection,
    r.last_malicious_detection = row.last_malicious_detection,

    r.first_quarantine_time = row.first_quarantine_time,
    r.last_quarantine_time = row.last_quarantine_time
"""


driver.execute_query(
    relationship_query,
    edges=edge_records,
    database_=DATABASE,
)

print(
    f"Imported {len(edge_records)} OBSERVED relationships."
)

Imported 72 OBSERVED relationships.


## Graph integrity validation

The following checks verify that the corrected graph contains:

- 247 endpoints
- 72 endpoint-artifact relationships
- artifact nodes corresponding to the relationship table
- explicit malicious evidence preserved on relationships

In [13]:
# Count Endpoint nodes.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)
    RETURN count(e) AS endpoint_count
    """,
    database_=DATABASE,
)

endpoint_count = result.records[0]["endpoint_count"]


# Count Artifact nodes.
result = driver.execute_query(
    """
    MATCH (a:Artifact)
    RETURN count(a) AS artifact_count
    """,
    database_=DATABASE,
)

artifact_count = result.records[0]["artifact_count"]


# Count OBSERVED relationships.
result = driver.execute_query(
    """
    MATCH ()-[r:OBSERVED]->()
    RETURN count(r) AS relationship_count
    """,
    database_=DATABASE,
)

relationship_count = result.records[0]["relationship_count"]


print("=" * 70)
print("GRAPH COUNTS")
print("=" * 70)

print("Endpoint nodes:", endpoint_count)
print("Artifact nodes:", artifact_count)
print("OBSERVED relationships:", relationship_count)

GRAPH COUNTS
Endpoint nodes: 247
Artifact nodes: 60
OBSERVED relationships: 72


In [14]:
# Confirm that every endpoint from the raw dataset exists in Neo4j.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)
    RETURN collect(e.endpoint_id) AS endpoints
    """,
    database_=DATABASE,
)

neo4j_endpoints = set(
    result.records[0]["endpoints"]
)

raw_endpoint_set = set(all_endpoint_ids)

missing_endpoints = sorted(
    raw_endpoint_set - neo4j_endpoints
)

unexpected_endpoints = sorted(
    neo4j_endpoints - raw_endpoint_set
)


print("=" * 70)
print("ENDPOINT COVERAGE")
print("=" * 70)

print("Raw endpoints:", len(raw_endpoint_set))
print("Neo4j endpoints:", len(neo4j_endpoints))
print("Missing endpoints:", len(missing_endpoints))
print("Unexpected endpoints:", len(unexpected_endpoints))

if missing_endpoints:
    print("\nMissing endpoints:")
    print(missing_endpoints)

if unexpected_endpoints:
    print("\nUnexpected endpoints:")
    print(unexpected_endpoints)

ENDPOINT COVERAGE
Raw endpoints: 247
Neo4j endpoints: 247
Missing endpoints: 0
Unexpected endpoints: 0


In [15]:
# Count endpoint-artifact relationships carrying explicit malicious evidence.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)
    WHERE r.malicious_detected = true

    RETURN
        count(r) AS malicious_relationship_count,
        count(DISTINCT e) AS malicious_endpoint_count,
        count(DISTINCT a) AS malicious_artifact_count
    """,
    database_=DATABASE,
)

malicious_summary = dict(result.records[0])


print("=" * 70)
print("MALICIOUS EVIDENCE VALIDATION")
print("=" * 70)

print(
    "Malicious relationships:",
    malicious_summary["malicious_relationship_count"],
)

print(
    "Malicious endpoints:",
    malicious_summary["malicious_endpoint_count"],
)

print(
    "Malicious artifacts:",
    malicious_summary["malicious_artifact_count"],
)

MALICIOUS EVIDENCE VALIDATION
Malicious relationships: 60
Malicious endpoints: 33
Malicious artifacts: 51


In [16]:
# Identify endpoints that exist in the graph but have no explicit malicious
# evidence on any of their observed artifact relationships.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)

    OPTIONAL MATCH (e)-[r:OBSERVED]->(a:Artifact)

    WITH
        e,
        collect(
            CASE
                WHEN r.malicious_detected = true
                THEN 1
                ELSE 0
            END
        ) AS malicious_flags

    RETURN
        count(
            CASE
                WHEN all(flag IN malicious_flags WHERE flag = 0)
                THEN e
            END
        ) AS endpoints_without_malicious_evidence
    """,
    database_=DATABASE,
)

print(
    "Endpoints without explicit malicious evidence:",
    result.records[0]["endpoints_without_malicious_evidence"],
)

Endpoints without explicit malicious evidence: 214


In [17]:
# Retrieve all final graph metrics in one query so the expected project
# population can be checked in a single validation step.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)
    WITH count(e) AS endpoints

    MATCH (a:Artifact)
    WITH
        endpoints,
        count(a) AS artifacts

    MATCH ()-[r:OBSERVED]->()
    WITH
        endpoints,
        artifacts,
        count(r) AS relationships

    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)
    WITH
        endpoints,
        artifacts,
        relationships,
        count(
            CASE
                WHEN r.malicious_detected = true
                THEN 1
            END
        ) AS malicious_relationships,
        count(
            DISTINCT CASE
                WHEN r.malicious_detected = true
                THEN e
            END
        ) AS malicious_endpoints

    RETURN
        endpoints,
        artifacts,
        relationships,
        malicious_relationships,
        malicious_endpoints
    """,
    database_=DATABASE,
)

final_metrics = dict(result.records[0])


print("=" * 70)
print("FINAL GRAPH VALIDATION")
print("=" * 70)

for key, value in final_metrics.items():
    print(f"{key}: {value}")


# These assertions protect the project from silently importing the wrong
# dataset or leaving stale graph data behind.
assert final_metrics["endpoints"] == 247, (
    "The graph does not contain the expected 247 endpoints."
)

assert final_metrics["relationships"] == 72, (
    "The graph does not contain the expected 72 OBSERVED relationships."
)

assert final_metrics["malicious_relationships"] == 60, (
    "The graph does not contain the expected 60 malicious relationships."
)

assert final_metrics["malicious_endpoints"] == 33, (
    "The graph does not contain the expected 33 malicious endpoints."
)

print("\nAll final graph validation checks passed.")

FINAL GRAPH VALIDATION
endpoints: 247
artifacts: 60
relationships: 72
malicious_relationships: 60
malicious_endpoints: 33

All final graph validation checks passed.


In [18]:
# Inspect one known endpoint with explicit malicious evidence.
# This provides a concrete forensic validation example for the project.
result = driver.execute_query(
    """
    MATCH (e:Endpoint {
        endpoint_id: 'EPVSWBNA'
    })
    OPTIONAL MATCH (e)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        a.sha256 AS sha256,
        a.file_names AS file_names,
        r.record_count AS record_count,
        r.malicious_detected AS malicious_detected,
        r.malicious_detection_count AS detection_count,
        r.malicious_file_names AS malicious_file_names,
        r.first_malicious_detection AS first_detection,
        r.last_malicious_detection AS last_detection

    ORDER BY r.first_malicious_detection
    """,
    database_=DATABASE,
)

epvswbna_df = pd.DataFrame(
    [dict(record) for record in result.records]
)

display(epvswbna_df)

,endpoint,sha256,file_names,record_count,malicious_detected,detection_count,malicious_file_names,first_detection,last_detection
0,EPVSWBNA,61a67e53c6c0c80a4e95af35a2997ad2ce59720020fc74...,[launcher.exe],1,True,1,[launcher.exe],2026-09-24T12:10:16.000000000,2026-09-24T12:10:16.000000000
1,EPVSWBNA,ae107170c153cb3eb34c049482722d3937f2c03a6c23ca...,[Strings.dll],1,True,1,[Strings.dll],2026-09-24T12:10:16.000000000,2026-09-24T12:10:16.000000000
2,EPVSWBNA,c6525c5230387653e16f21f408eab8202157b0eddeb180...,[zshp1020s.dll],1,True,1,[zshp1020s.dll],2026-09-24T12:10:16.000000000,2026-09-24T12:10:16.000000000


In [19]:
# Find an endpoint that exists in the raw population but has no explicit
# malicious evidence, demonstrating that the graph does not classify
# every endpoint as malicious.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)

    WHERE NOT EXISTS {
        MATCH (e)-[r:OBSERVED]->()
        WHERE r.malicious_detected = true
    }

    RETURN e.endpoint_id AS endpoint
    ORDER BY endpoint
    LIMIT 1
    """,
    database_=DATABASE,
)

if result.records:
    example_endpoint = result.records[0]["endpoint"]

    print(
        "Example endpoint without explicit malicious evidence:",
        example_endpoint,
    )
else:
    print(
        "No endpoint without explicit malicious evidence was found."
    )

Example endpoint without explicit malicious evidence: EP2C4ZH5


In [20]:
# Close the Neo4j driver cleanly after all import and validation work.
driver.close()

print("Neo4j driver closed.")

Neo4j driver closed.
